# Assignment 4 · Q-learning and PPO in MsPacMan

Select a Python kernel and run all cells from top to bottom. The notebook trains 30 main models and 45 mechanism models, evaluates them, and reconstructs the ten selected episodes. Allow several hours for a complete run.

**Environment.** The reported runs used Windows with Python 3.13.7, NumPy 2.5.4, pandas 3.1.1, Gymnasium 1.4.0, ALE 0.12.1, CPU PyTorch 2.14.1 and Stable-Baselines3 2.9.2. The kernel needs these packages and a working ALE MsPacMan installation.

**Outputs.** `setting_results` contains one row per trained agent and evaluation condition. `main_episodes` holds the 200 individual episodes for each of those rows, and `learning` holds the development results at the checkpoints. `mechanisms`, `mechanism_episodes` and `diagnostic` hold Part 3's per-model and per-episode results. `episode_roster` and `decision_log` hold the Part 4 cases.


In [ ]:
import copy
import hashlib
import platform
from time import perf_counter

import ale_py
import gymnasium as gym
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.policies import ActorCriticPolicy
from stable_baselines3.common.vec_env import DummyVecEnv
from torch import nn

gym.register_envs(ale_py)
torch.set_num_threads(1)

TRAIN_SEEDS = [17, 29, 101, 314, 2718]
BUDGET = 262_144

## Game rules and what the agents observe

An episode succeeds at 2,000 new points. It fails at game over or at the 1,000-decision horizon. A has deterministic action execution. B and C repeat the previous action with probability 0.25. A and B use the instructor-approved score increment, and C subtracts 0.1 per decision. Every policy chooses among NOOP and the four cardinal controls.

D uses the same game and freezes the B models. It filters object observations in the fixed region `[16, 48) × [24, 56)` before building the directional features. This tests observation robustness.

The environment code has three layers. The native task counts score and decisions. The feature encoder reads coordinates, available corridors, food and threat cues from the observations. The outer wrapper adds movement, lives and score information. The 61-value observation format stays compatible in D. The fixed mask changes what the agent sees while ALE executes the same game dynamics.


In [ ]:
MAZE_ROWS = ['111101111111101111', '100101000000101001', '111111111111111111', '010100100001001010', '110111111111111011', '010000100001000010', '011111100001111110', '010000100001000010', '110111111111111011', '010101010010101010', '111101011110101111', '100101110011101001', '100100010010001001', '111111111111111111']
PELLET_X = [8, 16, 24, 32, 40, 48, 56, 64, 72, 84, 92, 100, 108, 116, 124, 132, 140, 148]
PELLET_Y = [7, 19, 31, 43, 55, 67, 79, 91, 103, 115, 127, 139, 151, 163]
# Convert the pellet grid into a graph for distance and directional features.
GRID = np.array([[bit == '1' for bit in row] for row in MAZE_ROWS])
ALL_FOOD = np.array([(x, y) for y in PELLET_Y for x in PELLET_X], dtype=float) + [2, 1]

FOOD_BYTES = np.array([59, 60, 60, 60, 60, 61, 61, 61, 61,
                      60, 60, 60, 60, 61, 61, 61, 61, 59])
FOOD_BITS = np.array([64, 128, 32, 8, 2, 1, 4, 16, 64,
                     64, 16, 4, 1, 2, 8, 32, 128, 16], dtype=np.uint8)
FOOD_BYTES = (FOOD_BYTES[None, :] + 3 * np.arange(14)[:, None]).ravel()
FOOD_BITS = np.tile(FOOD_BITS, 14)

CELLS=np.flatnonzero(GRID.ravel())
XY=ALL_FOOD[CELLS]
LOOKUP={int(cell):i for i,cell in enumerate(CELLS)}
NEIGHBORS=np.full((len(CELLS),4),-1,dtype=int)
for i,cell in enumerate(CELLS):
    r,c=divmod(int(cell),18)
    for action,(dr,dc) in enumerate([(-1,0),(0,1),(0,-1),(1,0)]):
        if 0<=r+dr<14 and 0<=c+dc<18:
            NEIGHBORS[i,action]=LOOKUP.get((r+dr)*18+c+dc,-1)
DISTANCE=np.full((len(CELLS),len(CELLS)),1000,dtype=np.float32)
np.fill_diagonal(DISTANCE,0)
for i,n in enumerate(NEIGHBORS):DISTANCE[i,n[n>=0]]=1
for k in range(len(CELLS)):DISTANCE=np.minimum(DISTANCE,DISTANCE[:,k,None]+DISTANCE[None,k,:])
assert np.max(DISTANCE)<1000
POWER_NODES=np.argmin(np.square(XY[:,None,:]-np.array([[150,150],[150,18],[10,150],[10,18]])[None,:,:]).sum(axis=2),axis=0)

In [ ]:
def native_game(setting):
    return gym.make("ALE/MsPacman-v5", obs_type="ram", frameskip=4, mode=0,
                    repeat_action_probability=0.0 if setting == "A" else 0.25)


class RawTask(gym.Wrapper):
    def __init__(self, setting):
        super().__init__(native_game(setting))
        self.setting = setting
        self.action_space = gym.spaces.Discrete(5)
        self.observation_space=gym.spaces.Box(0,255,shape=(128,),dtype=np.uint8)

    def reset(self,*,seed=None,options=None):
        _,info=self.env.reset(seed=seed,options=options)
        self.unwrapped.ale.setRAM(0,0)
        self.ram=self.unwrapped.ale.getRAM().copy()
        self.steps,self.score,self.episode_return=0,0.,0.
        self.finished = False
        return self.ram.copy(),dict(info)

    def step(self,action):
        if self.finished:raise RuntimeError('Call reset() after a terminal episode.')
        ram,raw_reward,terminal,truncated,info=self.env.step(int(action))
        self.steps+=1;self.score+=float(raw_reward)
        success=self.score>=2000
        timeout=self.steps>=1000 and not success
        if int(ram[0])!=0:
            raise RuntimeError('The episode left the registered maze.')
        self.ram=ram.copy()
        reward=float(raw_reward)-(0.1 if self.setting=='C' else 0)
        self.episode_return+=reward
        self.finished=bool(terminal or truncated or success or timeout)
        reason='success' if success else ('time_limit' if timeout else ('game_over' if terminal else 'external_limit'))
        info=dict(info,score=self.score,raw_reward=float(raw_reward),success=bool(success),
            episode_steps=self.steps,reason=reason,task_return=self.episode_return,maze=int(ram[0]))
        return self.ram.copy(),reward,bool(terminal or success or timeout),bool(truncated),info

In [ ]:
class RichTask(gym.Wrapper):
    def __init__(self, setting):
        super().__init__(RawTask(setting))
        self.observation_space=gym.spaces.Box(-1,1,shape=(45,),dtype=np.float32)
        self.previous_player=None
        self.previous_ghosts=None
        self.food_key=None

    def observation(self):
        ram=self.env.ram
        assert int(ram[0])==0
        player=np.array([float(ram[10])-8,float(ram[16])+6])
        ghosts=np.column_stack((ram[6:10].astype(float)-8,ram[12:16].astype(float)+6))
        active=np.arange(4)<=int(ram[19])
        if not ram[47]:active[:]=False
        gd=np.abs(ghosts-player).sum(axis=1)
        gd[~active]=np.inf
        nearest=int(np.argmin(gd))
        ghost_delta=ghosts[nearest]-player if active.any() else np.zeros(2)
        ghost_distance=float(gd[nearest]) if active.any() else 320.
        present=GRID.ravel()&((ram[FOOD_BYTES]&FOOD_BITS)!=0)
        # Power pills occupy the same topological corridors but separate RAM bits.
        power=np.array([bool(int(ram[117])&bit) for bit in (4,8,16,32)])
        food=ALL_FOOD[present]
        if len(food):
            delta=food-player;distance=np.abs(delta).sum(axis=1);j=int(np.argmin(distance))
            pellet_delta,pellet_distance=delta[j],float(distance[j])
            direction=np.where(np.abs(delta[:,0])>np.abs(delta[:,1]),
                np.where(delta[:,0]>0,1,3),np.where(delta[:,1]>0,2,0))
            density=np.bincount(direction,minlength=4)/252
        else:pellet_delta,pellet_distance,density=np.zeros(2),320.,np.zeros(4)
        node=int(np.argmin(np.square((XY-player)/[8,12]).sum(axis=1)))
        neighbors=NEIGHBORS[node];available=neighbors>=0
        key=ram[59:101].tobytes()+bytes([int(ram[117])&60])
        if key!=self.food_key:
            targets=present[CELLS].copy();targets[POWER_NODES[power]]=True
            self.food_dist=DISTANCE[:,targets].min(axis=1) if targets.any() else np.full(len(CELLS),32.)
            self.power_dist=DISTANCE[:,POWER_NODES[power]].min(axis=1) if power.any() else np.full(len(CELLS),32.)
            self.food_key=key
        def regret(dist):
            costs=np.full(4,32.);costs[available]=1+dist[neighbors[available]]
            out=np.clip((costs-costs[available].min())/8,0,1);out[~available]=1
            return out
        threat=np.zeros(4)
        if active.any():
            ghost_nodes=np.argmin(np.square((XY[:,None,:]-ghosts[None,:,:])/[8,12]).sum(axis=2),axis=0)[active]
            path_distance=DISTANCE[:,ghost_nodes].min(axis=1)
            threat[available]=np.exp(-path_distance[neighbors[available]]/3)
        motion=np.zeros((4,2)) if self.previous_ghosts is None else ghosts-self.previous_ghosts
        player_motion=np.zeros(2) if self.previous_player is None else player-self.previous_player
        if np.linalg.norm(player_motion)>20:player_motion[:]=0
        motion[np.abs(motion)>20]=0
        self.previous_ghosts,self.previous_player=ghosts.copy(),player.copy()
        base=[player[0]/160,player[1]/210,ghost_delta[0]/160,ghost_delta[1]/210,ghost_distance/320,
            pellet_delta[0]/160,pellet_delta[1]/210,pellet_distance/320,*density,
            min(int(ram[116]),63)/63,(int(ram[56])&3)/3,max(0,1-self.env.steps/1000)]
        offset=np.clip((player-XY[node])/[8,12],-1,1)
        obs=np.array([*base,*available.astype(float),*regret(self.food_dist),*threat,*regret(self.power_dist),
            *offset,*np.clip(motion.ravel()/8,-1,1),*np.clip(player_motion/8,-1,1),
            present.sum()/154,1/(1+np.linalg.norm(player-XY[node]))],dtype=np.float32)
        self.node=node
        return np.clip(obs,-1,1)

    def reset(self,*,seed=None,options=None):
        _,info=self.env.reset(seed=seed,options=options)
        self.previous_player=self.previous_ghosts=self.food_key=None
        return self.observation(),info

    def step(self,action):
        _,reward,term,trunc,info=self.env.step(int(action))
        return self.observation(),reward,term,trunc,info

In [ ]:
class GameTask(gym.Wrapper):
    def __init__(self, setting):
        super().__init__(RichTask(setting))
        self.observation_space = gym.spaces.Box(-1, 1, shape=(61,), dtype=np.float32)

    def extend(self,obs,info):
        ram=self.env.env.ram
        dx=ram[6:10].astype(float)-float(ram[10]);dy=ram[12:16].astype(float)-float(ram[16])
        active=(np.arange(4)<=int(ram[19]))&bool(ram[47])
        distances=np.abs(dx)+np.abs(dy);distances[~active]=np.inf
        result=np.empty(61,dtype=np.float32);result[:45]=obs
        result[45:47]=obs[33:41].reshape(4,2)[distances.argmin()] if active.any() else 0
        result[47]=min(self.env.env.score/2000,1);result[48]=info['lives']/5
        offsets=result[49:57].reshape(4,2)
        offsets[:,0]=np.clip(dx/160,-1,1);offsets[:,1]=np.clip(dy/210,-1,1);offsets[~active]=0
        result[57:61]=active
        return result

    def _key(self):
        ram=self.env.env.ram
        return (ram[6:11].tobytes()+ram[12:17].tobytes()+bytes(int(ram[i]) for i in [0,19,47,56])
            +ram[59:101].tobytes()+ram[116:118].tobytes())
    def reset(self,*,seed=None,options=None):
        self._last_key=None;self._last_observation=None
        features, info = self.env.reset(seed=seed, options=options)
        obs = self.extend(features, info)
        self._last_key=self._key();self._last_observation=obs.copy()
        return obs,info
    def step(self,action):
        _,reward,term,trunc,info=self.env.env.step(int(action))
        key=self._key()
        if self._last_key==key and not np.any(self._last_observation[33:43]):
            obs=self._last_observation.copy()
            obs[14]=max(0,1-self.env.env.steps/1000)
            obs[47]=min(self.env.env.score/2000,1);obs[48]=info['lives']/5
        else:obs=self.extend(self.env.observation(),info)
        self._last_key=key;self._last_observation=obs.copy()
        return obs,reward,term,trunc,info

In [ ]:
ROI = (16, 24, 48, 56)


def inside(points,roi=ROI):
    points=np.asarray(points)
    return ((points[...,0]>=roi[0])&(points[...,0]<roi[2])&
            (points[...,1]>=roi[1])&(points[...,1]<roi[3]))


class ObservationShift(gym.Wrapper):
    def __init__(self,occluded=True):
        super().__init__(GameTask('B'))
        self.occluded=occluded
        self.last_player=self.last_orientation=self.previous_player=self.previous_ghosts=None
        self.food_key=None

    def encode(self,ram,info):
        actual_player=np.array([float(ram[10])-8,float(ram[16])+6])
        # Retain the last visible player pose while the mask hides the avatar.
        player_hidden=bool(self.occluded and inside(actual_player))
        if not player_hidden:
            self.last_player=actual_player.copy();self.last_orientation=(int(ram[56])&3)/3
        if self.last_player is None:
            raise RuntimeError('The predefined mask must not hide the initial player pose.')
        player=self.last_player.copy()
        ghosts=np.column_stack((ram[6:10].astype(float)-8,ram[12:16].astype(float)+6))
        hidden=inside(ghosts) if self.occluded else np.zeros(4,dtype=bool)
        # Use only visible ghosts in the nearest-ghost and corridor threat features.
        active=(np.arange(4)<=int(ram[19]))&bool(ram[47])&~hidden
        # Hidden coordinates never influence motion, nearest threat or offsets.
        ghosts[hidden]=self.previous_ghosts[hidden] if self.previous_ghosts is not None else 0
        gd=np.abs(ghosts-player).sum(axis=1);gd[~active]=np.inf
        nearest=int(np.argmin(gd));ghost_delta=ghosts[nearest]-player if active.any() else np.zeros(2)
        ghost_distance=float(gd[nearest]) if active.any() else 320.
        present=GRID.ravel()&((ram[FOOD_BYTES]&FOOD_BITS)!=0)
        if self.occluded:present[inside(ALL_FOOD)]=False
        power=np.array([bool(int(ram[117])&bit) for bit in (4,8,16,32)])
        # The four power-pellet centres are all outside this fixed region.
        food=ALL_FOOD[present]
        if len(food):
            delta=food-player;distance=np.abs(delta).sum(axis=1);j=int(np.argmin(distance))
            pellet_delta,pellet_distance=delta[j],float(distance[j])
            direction=np.where(np.abs(delta[:,0])>np.abs(delta[:,1]),np.where(delta[:,0]>0,1,3),np.where(delta[:,1]>0,2,0))
            density=np.bincount(direction,minlength=4)/252
        else:pellet_delta,pellet_distance,density=np.zeros(2),320.,np.zeros(4)
        node=int(np.argmin(np.square((XY-player)/[8,12]).sum(axis=1)))
        neighbors=NEIGHBORS[node];available=neighbors>=0
        key=present.tobytes()+power.tobytes()
        if key!=self.food_key:
            targets=present[CELLS].copy();targets[POWER_NODES[power]]=True
            self.food_dist=DISTANCE[:,targets].min(axis=1) if targets.any() else np.full(len(CELLS),32.)
            self.power_dist=DISTANCE[:,POWER_NODES[power]].min(axis=1) if power.any() else np.full(len(CELLS),32.)
            self.food_key=key
        def regret(dist):
            costs=np.full(4,32.);costs[available]=1+dist[neighbors[available]]
            out=np.clip((costs-costs[available].min())/8,0,1);out[~available]=1
            return out
        threat=np.zeros(4)
        if active.any():
            ghost_nodes=np.argmin(np.square((XY[:,None,:]-ghosts[None,:,:])/[8,12]).sum(axis=2),axis=0)[active]
            path_distance=DISTANCE[:,ghost_nodes].min(axis=1)
            threat[available]=np.exp(-path_distance[neighbors[available]]/3)
        motion=np.zeros((4,2)) if self.previous_ghosts is None else ghosts-self.previous_ghosts
        player_motion=np.zeros(2) if self.previous_player is None else player-self.previous_player
        if np.linalg.norm(player_motion)>20:player_motion[:]=0
        motion[np.abs(motion)>20]=0
        if self.occluded:motion[~active]=0
        self.previous_ghosts,self.previous_player=ghosts.copy(),player.copy()
        steps=info['episode_steps'] if 'episode_steps' in info else 0
        base=[player[0]/160,player[1]/210,ghost_delta[0]/160,ghost_delta[1]/210,ghost_distance/320,
              pellet_delta[0]/160,pellet_delta[1]/210,pellet_distance/320,*density,
              min(int(ram[116]),63)/63,self.last_orientation,max(0,1-steps/1000)]
        offset=np.clip((player-XY[node])/[8,12],-1,1)
        raw=np.array([*base,*available.astype(float),*regret(self.food_dist),*threat,*regret(self.power_dist),
                      *offset,*np.clip(motion.ravel()/8,-1,1),*np.clip(player_motion/8,-1,1),present.sum()/154,
                      1/(1+np.linalg.norm(player-XY[node]))],dtype=np.float32)
        raw=np.clip(raw,-1,1)
        nearest_motion=raw[33:41].reshape(4,2)[nearest] if active.any() else np.zeros(2)
        extra=(ghosts-player)/[160,210];extra=np.clip(extra,-1,1);extra[~active]=0
        self.hidden_player,self.hidden_ghosts=player_hidden,int(hidden.sum())
        score=info.get('score',0.)
        return np.r_[raw,nearest_motion,min(score/2000,1),info['lives']/5,extra.ravel(),active].astype(np.float32)

    def reset(self,*,seed=None,options=None):
        original,info=self.env.reset(seed=seed,options=options)
        self.last_player=self.last_orientation=self.previous_player=self.previous_ghosts=self.food_key=None
        obs=self.encode(self.unwrapped.ale.getRAM(),info)
        if not self.occluded:assert np.array_equal(obs,original),'Unoccluded encoder parity'
        return obs,dict(info,observation_occluded=self.occluded)

    def step(self,action):
        original,reward,term,trunc,info=self.env.step(action)
        obs=self.encode(self.unwrapped.ale.getRAM(),info)
        if not self.occluded:assert np.array_equal(obs,original),'Unoccluded encoder parity'
        return obs,reward,term,trunc,dict(info,observation_occluded=self.occluded,
                                       hidden_player=self.hidden_player,hidden_ghosts=self.hidden_ghosts)

## How Q-learning and PPO learn

Q-learning makes one Bellman update per observed transition and uses a food-relative discrete key. PPO uses the same observations, a shared directional action scorer, a separate value network and clipped on-policy updates. Both scale rewards by 100 inside the learning rule. Frozen evaluation uses greedy Q with seeded ties or PPO argmax. The Part 3 diagnostic also tests PPO sampling.

`train_agent` returns frozen copies at each checkpoint, the selected interaction count, development results and training time. `evaluate` returns one row per episode. `score` is the increase in game points. `task_return` includes the step penalty in C. The success flag always uses the 2,000-point game-score threshold. Training reward scaling is internal to the learners.


In [ ]:
class TabularQ:
    """One table, one Bellman update per newly observed transition.

    The food-relative discrete key and world-action mapping match the reported
    selected Q configuration. There is no replay, second Q table or action mask.
    """
    ACTIONS = np.array([1, 2, 4, 3])  # Clockwise UP, RIGHT, DOWN, LEFT -> ALE ID.
    DIRECTIONS = np.array([[0., -1.], [1., 0.], [0., 1.], [-1., 0.]])

    def __init__(self, seed, alpha=.15, gamma=.995):
        self.rng = np.random.default_rng(seed)
        self.alpha, self.gamma = float(alpha), float(gamma)
        self.table = {}
        self.update_calls = 0

    def state(self, obs):
        """Coarsen common observed food, corridors, threat, motion and power."""
        order = [0, 1, 3, 2]
        available = obs[15:19][order] > .5
        costs = obs[19:23][order].copy()
        costs[~available] = 2
        distance = float(obs[7] * 320)
        delta = obs[5:7] * np.array([160 / 8, 210 / 12])
        projection = self.DIRECTIONS @ delta
        near = distance <= 12 and np.any((projection > 0) & available)
        if near:
            projection[~available] = -100
            anchor = int(np.argmax(projection))
            best = (projection == projection[anchor]) & available
        else:
            anchor = int(np.argmin(costs))
            best = (costs == costs.min()) & available
        def bits(values):
            return sum(int(value) << i for i, value in enumerate(np.roll(values, -anchor)))
        danger = (obs[23:27][order] > .36) & available
        motion = obs[41:43]
        moving = 4 if np.linalg.norm(motion) < .01 else (int(np.argmax(self.DIRECTIONS @ motion)) - anchor) % 4
        key = (bits(available), bits(best), bits(danger), int(obs[12] > 0),
               int(np.searchsorted([3., 12.], distance)), moving, int(np.max(np.abs(obs[31:33])) < .2))
        return anchor, key

    def values(self, state):
        anchor, key = state
        canonical = self.table.get(key, np.zeros(5, dtype=np.float32))
        world = np.empty(5, dtype=np.float32)
        world[0] = canonical[0]
        world[self.ACTIONS[(np.arange(4) + anchor) % 4]] = canonical[1:]
        return world

    def values_for_observation(self, obs):
        return self.values(self.state(obs))

    def action(self, state, epsilon=0.):
        if self.rng.random() < epsilon:
            return int(self.rng.integers(5))
        values = self.values(state)
        return int(self.rng.choice(np.flatnonzero(values == values.max())))

    def update(self, state, action, reward, next_state, terminal):
        anchor, key = state
        index = 0 if action == 0 else 1 + (int(np.flatnonzero(self.ACTIONS == action)[0]) - anchor) % 4
        values = self.table.setdefault(key, np.zeros(5, dtype=np.float32))
        # Q(s,a) <- Q(s,a) + alpha * [r/100 + gamma*max Q(s',.) - Q(s,a)]
        target = float(reward) / 100
        if not terminal:
            target += self.gamma * float(self.table.get(next_state[1], np.zeros(5, dtype=np.float32)).max())
        values[index] += self.alpha * (target - float(values[index]))
        self.update_calls += 1

    def fingerprint(self):
        digest = hashlib.sha256(str((self.alpha, self.gamma, self.update_calls)).encode())
        for key, values in sorted(self.table.items()):
            digest.update(np.asarray(key, dtype=np.int64).tobytes())
            digest.update(values.tobytes())
        return digest.hexdigest()

In [ ]:
class ActionScorer(nn.Module):
    """Shared 23-feature action scorer and separate 49-input value network.

    Each of the five available actions is expressed in relative coordinates
    and scored by the same two-layer network. No action is forced or masked.
    The selected policy uses the first 49 of the common 61 observed values.
    """
    latent_dim_pi = 5
    latent_dim_vf = 64

    def __init__(self):
        super().__init__()
        self.scorer = nn.Sequential(nn.Linear(23, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1))
        self.value_net = nn.Sequential(nn.Linear(49, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh())
        self.register_buffer("directions", torch.tensor([[0., 0.], [0., -1.], [1., 0.], [-1., 0.], [0., 1.]]))

    def actor_features(self, obs):
        direction = self.directions
        perpendicular = torch.stack((-direction[:, 1], direction[:, 0]), dim=1)
        ones = torch.ones_like(obs[:, :1])
        global_values = obs[:, [4, 7, 12, 14, 43, 47, 48]].unsqueeze(1).expand(-1, 5, -1)
        noop = torch.zeros((len(obs), 5), device=obs.device)
        noop[:, 0] = 1
        columns = [torch.cat((ones, obs[:, 15:19]), dim=1),
                   torch.cat((ones, obs[:, 19:23]), dim=1),
                   torch.cat((obs[:, 23:27].max(dim=1, keepdim=True).values, obs[:, 23:27]), dim=1),
                   torch.cat((ones, obs[:, 27:31]), dim=1), noop]
        for pair in (obs[:, 41:43], obs[:, 2:4], obs[:, 5:7], obs[:, 31:33], obs[:, 45:47]):
            columns.extend([pair @ direction.T, pair @ perpendicular.T])
        columns.append(torch.cat((torch.zeros_like(ones), obs[:, [8, 9, 11, 10]]), dim=1))
        return torch.cat((global_values, torch.stack(columns, dim=2)), dim=2)

    def forward_actor(self, obs):
        return self.scorer(self.actor_features(obs)).squeeze(-1)

    def forward_critic(self, obs):
        return self.value_net(obs[:, :49])

    def forward(self, obs):
        return self.forward_actor(obs), self.forward_critic(obs)


class SharedActionPolicy(ActorCriticPolicy):
    def _build_mlp_extractor(self):
        self.mlp_extractor = ActionScorer().to(self.device)

    def _build(self, lr_schedule):
        super()._build(lr_schedule)
        self.action_net = nn.Identity()
        nn.init.orthogonal_(self.mlp_extractor.scorer[-1].weight, .01)
        nn.init.zeros_(self.mlp_extractor.scorer[-1].bias)
        self.optimizer = self.optimizer_class(self.parameters(), lr=lr_schedule(1), **self.optimizer_kwargs)


class FrozenPolicy:
    """Read-only highest-probability evaluation, with learned-weight fingerprint."""
    def __init__(self, policy):
        self.policy = policy

    def values_for_observation(self, obs):
        with torch.no_grad():
            distribution = self.policy.get_distribution(torch.as_tensor(obs[None], device=self.policy.device))
            return distribution.distribution.probs.cpu().numpy()[0].copy()

    def fingerprint(self):
        return hashlib.sha256(b"".join(value.detach().cpu().numpy().tobytes()
                                      for value in self.policy.state_dict().values())).hexdigest()


class RewardScale(gym.RewardWrapper):
    def reward(self, reward):
        return reward / 100


class Checkpoints(BaseCallback):
    def __init__(self, budget, points, save, entropy_start, entropy_end):
        super().__init__()
        self.budget, self.points, self.save = budget, set(points), save
        self.entropy_start, self.entropy_end = entropy_start, entropy_end

    def _on_rollout_start(self):
        # Save/evaluate at rollout boundaries, after the previous PPO update.
        if self.num_timesteps in self.points:
            self.save(self.num_timesteps)

    def _on_step(self):
        self.model.ent_coef = self.entropy_end + (0.018 if self.entropy_start else 0.0) * max(0, 1 - self.num_timesteps / self.budget)
        return True


def build_model(envs, seed, **overrides):
    """Reported stable configuration; all settings stay fixed across A/B/C."""
    settings = dict(learning_rate=lambda progress: .00005 + (.0003 - .00005) * max(0, progress),
                    gamma=.995, gae_lambda=.97, ent_coef=.02, n_steps=512, batch_size=256,
                    n_epochs=6, clip_range=.15, target_kl=.015, seed=seed, device="cpu", verbose=0)
    settings.update(overrides)
    return PPO(SharedActionPolicy, envs, **settings)

In [ ]:
def frozen_action(values, algorithm, rng, sampled=False):
    if algorithm == "Q-learning":
        rng.random()  # Match the training policy RNG's epsilon check.
        return int(rng.choice(np.flatnonzero(values == values.max())))
    if not sampled:
        return int(values.argmax())
    probabilities = values.astype(float)
    probabilities /= probabilities.sum()
    return int(rng.choice(5, p=probabilities))


def evaluate(agent, algorithm, setting, seeds, obstructed=None, sampled=False):
    # Compare the learned parameters before and after the evaluation.
    before = agent.fingerprint()
    environment = GameTask(setting) if obstructed is None else ObservationShift(obstructed)
    rows = []
    try:
        for seed in seeds:
            observation, _ = environment.reset(seed=int(seed))
            # Give policy tie-breaking or sampling its own seeded RNG.
            rng = np.random.default_rng(int(seed) + 3_000_000)
            entropy = disagreements = 0
            for step in range(1, 1001):
                values = agent.values_for_observation(observation)
                action = frozen_action(values, algorithm, rng, sampled)
                if algorithm == "PPO":
                    greedy = int(values.argmax())
                    distribution = values.astype(float)
                    distribution /= distribution.sum()
                    entropy -= float(np.sum(distribution * np.log(distribution + 1e-30)))
                    disagreements += action != greedy
                observation, reward, terminated, truncated, info = environment.step(action)
                if terminated or truncated:
                    row = dict(evaluation_seed=int(seed), success=int(info["success"]), length=step,
                               score=info["score"], task_return=info["task_return"])
                    if algorithm == "PPO":
                        row.update(mean_action_entropy=entropy / step,
                                   sampled_disagreement=disagreements / step)
                    rows.append(row)
                    break
    finally:
        environment.close()
    assert agent.fingerprint() == before
    assert len(rows) == len(seeds)
    return pd.DataFrame(rows)


def train_agent(algorithm, setting, seed, config, development_seeds, budget=BUDGET):
    checkpoints = {0, budget, *range(65_536, budget, 65_536)}
    scores, agents = [], {}
    training_seconds = 0.

    def assess(point, frozen):
        # Development episodes run separately from the training environment.
        results = evaluate(frozen, algorithm, setting, development_seeds)
        scores.append((point, results.success.mean(), results.score.mean()))
        agents[point] = frozen

    if algorithm == "Q-learning":
        model = TabularQ(seed, config["alpha"], config["gamma"])
        env = GameTask(setting)
        try:
            episode = 0
            obs, _ = env.reset(seed=seed * 1000)
            assess(0, copy.deepcopy(model))
            clock = perf_counter()
            for index in range(budget):
                state = model.state(obs)
                epsilon = max(.05, 1 - .95 * index / (config["epsilon_fraction"] * budget))
                action = model.action(state, epsilon)
                obs, reward, terminated, truncated, _ = env.step(action)
                # One Bellman update uses this transition and the episode boundary.
                model.update(state, action, reward, model.state(obs), terminated or truncated)
                if terminated or truncated:
                    episode += 1
                    obs, _ = env.reset(seed=seed * 1000 + episode)
                if index + 1 in checkpoints:
                    training_seconds += perf_counter() - clock
                    assess(index + 1, copy.deepcopy(model))
                    clock = perf_counter()
        finally:
            env.close()
        assert model.update_calls == budget
    else:
        keys = ("score", "success", "reason", "episode_steps")
        # Four environments supply 512 decisions each to a PPO rollout.
        envs = DummyVecEnv([lambda: Monitor(RewardScale(GameTask(setting)), info_keywords=keys)
                            for _ in range(4)])
        start, end = config["lr_start"], config["lr_end"]
        try:
            model = build_model(envs, seed, gamma=config["gamma"], n_epochs=config["epochs"],
                                ent_coef=config["entropy_start"],
                                learning_rate=lambda progress: end + (start - end) * max(0, progress))
            clock = perf_counter()

            def checkpoint(point):
                nonlocal clock, training_seconds
                training_seconds += perf_counter() - clock
                assert model.num_timesteps == point
                # Construct a frozen copy without moving the training RNG.
                with torch.random.fork_rng(devices=[]):
                    policy = SharedActionPolicy(envs.observation_space, envs.action_space, lambda _: 0.0)
                policy.load_state_dict(model.policy.state_dict())
                policy.set_training_mode(False)
                assess(point, FrozenPolicy(policy))
                clock = perf_counter()

            callback = Checkpoints(budget, checkpoints - {budget}, checkpoint,
                                   config["entropy_start"], config["entropy_end"])
            model.learn(budget, callback=callback)
            checkpoint(budget)
        finally:
            envs.close()
    # Rank positive checkpoints by development success, score and interaction count.
    selected = max((row for row in scores if row[0]), key=lambda row: (row[1], row[2], -row[0]))[0]
    development = pd.DataFrame(scores, columns=["interactions", "success_rate", "mean_score"])
    return agents, selected, development, training_seconds

In [ ]:
ARMS={
 'Q_base':dict(algorithm='Q-learning',alpha=.15,gamma=.995,epsilon_fraction=.8),
 'Q_fast_epsilon':dict(algorithm='Q-learning',alpha=.15,gamma=.995,epsilon_fraction=.2),
 'Q_gamma99':dict(algorithm='Q-learning',alpha=.15,gamma=.99,epsilon_fraction=.8),
 'Q_alpha05':dict(algorithm='Q-learning',alpha=.05,gamma=.995,epsilon_fraction=.8),
 'PPO_base':dict(algorithm='PPO',gamma=.995,entropy_start=.02,entropy_end=.002,lr_start=.0003,lr_end=.00005,epochs=6),
 'PPO_gamma99':dict(algorithm='PPO',gamma=.99,entropy_start=.02,entropy_end=.002,lr_start=.0003,lr_end=.00005,epochs=6),
 'PPO_entropy0':dict(algorithm='PPO',gamma=.995,entropy_start=0.,entropy_end=0.,lr_start=.0003,lr_end=.00005,epochs=6),
 'PPO_lr1e4':dict(algorithm='PPO',gamma=.995,entropy_start=.02,entropy_end=.002,lr_start=.0001,lr_end=.00005,epochs=6),
 'PPO_epochs2':dict(algorithm='PPO',gamma=.995,entropy_start=.02,entropy_end=.002,lr_start=.0003,lr_end=.00005,epochs=2),
}

## Performance across the four experimental settings

The common 262,144-interaction budget gives PPO 128 rollouts of 2,048 interactions each. A/B/C use the selected hyperparameters throughout. The matched development search compared seven packages per method and repeated the best two. This experiment repeats the selected configurations. Checkpoints are chosen by success, then score, then the earlier checkpoint on ten separate development seeds. Five training seeds are used for each condition and algorithm.

Each frozen model is evaluated on 200 held-out seeds. D evaluates the same B weights with clean and obstructed observations on a separate common seed list. The run records training and evaluation wall time on the current machine.

| Phase | Episode seeds | Use |
| --- | --- | --- |
| A/B/C development | 1130000–1130009 | Choose checkpoints within each training run |
| A/B/C final evaluation | 1150000–1150199 | Measure success, return and episode length |
| Clean and obstructed D | 1600000–1600199 | Compare the two observations for each frozen B agent |

The five training seeds are 17, 29, 101, 314 and 2718. A final result row describes one trained agent. Its success rate is the number of successful episodes divided by 200.


In [ ]:
def evaluate_run(agent, algorithm, setting, seed, seeds, training_interactions,
                 checkpoint, training_seconds, obstructed=None, sampled=False):
    clock = perf_counter()
    episodes = evaluate(agent, algorithm, setting, seeds, obstructed=obstructed, sampled=sampled)
    evaluation_seconds = perf_counter() - clock
    success = episodes.success.eq(1)
    penalty = .1 if setting == "C" else 0.
    assert np.array_equal(episodes.success, episodes.score.ge(2000))
    np.testing.assert_allclose(episodes.task_return, episodes.score - penalty * episodes.length,
                               atol=1e-8)
    return dict(algorithm=algorithm, seed=seed, episodes=len(episodes),
                success_rate=success.mean(), failure_rate=(~success).mean(),
                mean_return=episodes.task_return.mean(),
                success_length=episodes.loc[success, "length"].mean(),
                failure_length=episodes.loc[~success, "length"].mean(),
                training_interactions=training_interactions, checkpoint=checkpoint,
                training_seconds=training_seconds, evaluation_seconds=evaluation_seconds), episodes


def compare_settings(training_seeds, development_seeds, final_seeds, shift_seeds, budget=BUDGET):
    models, episodes, curves, rows = {}, {}, [], []
    for setting in "ABC":
        for arm in ("Q_base", "PPO_base"):
            config = ARMS[arm]
            algorithm = config["algorithm"]
            for seed in training_seeds:
                trained, selected, development, seconds = train_agent(
                    algorithm, setting, seed, config, development_seeds, budget)
                model = trained[selected]
                key = (algorithm, setting, seed)
                row, outcomes = evaluate_run(model, algorithm, setting, seed, final_seeds,
                                             budget, selected, seconds)
                rows.append(dict(row, condition=setting))
                curves.append(development.assign(algorithm=algorithm, condition=setting, seed=seed))
                models[key] = model
                episodes[key] = outcomes
                # Pair both D evaluations with the identical B checkpoint and seeds.
                if setting == "B":
                    for mask, label in ((False, "D clean"), (True, "D obstructed")):
                        row, outcomes = evaluate_run(model, algorithm, "B", seed, shift_seeds,
                                                     0, selected, 0., obstructed=mask)
                        rows.append(dict(row, condition=label))
                        episodes[algorithm, label, seed] = outcomes
                    assert episodes[algorithm, "D clean", seed].evaluation_seed.equals(
                        episodes[algorithm, "D obstructed", seed].evaluation_seed)
    return models, episodes, pd.concat(curves, ignore_index=True), pd.DataFrame(rows)

In [ ]:
main_models, main_episodes, learning, setting_results = compare_settings(
    TRAIN_SEEDS, range(1_130_000, 1_130_010), range(1_150_000, 1_150_200),
    range(1_600_000, 1_600_200))

print(platform.platform(), platform.processor(), f"PyTorch {torch.__version__}, CPU, one Torch thread")
display(setting_results)

## How training choices affect performance

Each arm changes one factor relative to its baseline in B. All training budgets, seeds, and final evaluation procedures are matched. This part uses each model's final weights. The diagnostic evaluates the two entropy arms with both argmax and sampled actions on the same 50 seeds.

The development seeds are 1480000–1480009. The primary test uses 1500000–1500199. The separate diagnostic uses 1700000–1700049 and records mean action entropy and the fraction of sampled actions that differ from argmax in each episode.


In [ ]:
def compare_mechanisms(training_seeds, development_seeds, final_seeds, diagnostic_seeds, budget=BUDGET):
    results, episodes, diagnostics = [], [], []
    for arm, config in ARMS.items():
        algorithm = config["algorithm"]
        for seed in training_seeds:
            checkpoints, _, _, seconds = train_agent(
                algorithm, "B", seed, config, development_seeds, budget)
            # The mechanism comparison uses full-budget checkpoints.
            agent = checkpoints[budget]
            row, outcomes = evaluate_run(agent, algorithm, "B", seed, final_seeds, budget, budget, seconds)
            results.append(dict(row, arm=arm))
            episodes.append(outcomes.assign(arm=arm, seed=seed))
            if arm in ("PPO_base", "PPO_entropy0"):
                for sampled in (False, True):
                    outcomes = evaluate(agent, algorithm, "B", diagnostic_seeds, sampled=sampled)
                    diagnostics.append(outcomes.assign(arm=arm, seed=seed,
                                                       mode="sampled" if sampled else "argmax"))
    return pd.DataFrame(results), pd.concat(episodes, ignore_index=True), pd.concat(diagnostics, ignore_index=True)

In [ ]:
mechanisms, mechanism_episodes, diagnostic = compare_mechanisms(
    TRAIN_SEEDS, range(1_480_000, 1_480_010), range(1_500_000, 1_500_200), range(1_700_000, 1_700_050))
display(mechanisms)

## Decisions behind successes and failures

Typical episodes are near the outcome-conditional median length and score. For a surprising outcome, use the model for which the outcome has the lowest positive frequency, then choose its conditional median. The ten strata cover three typical successes, three typical failures, two surprising successes and two surprising failures.

Recreate each selected episode with its frozen policy and seed. For each of the three decisions, replay the identical action prefix in a second environment before trying a different action. Results reflect that realized transition. They do not establish expected risk or the outcome of a complete alternative route.

The decision log records the five Q-values or action probabilities, the requested action, the resulting position and reward, and the tested alternative. Lives are read from the emulator. In a sticky setting, the emulator may repeat the previous input, so the requested action alone does not determine the transition.


In [ ]:
LABELS=['N','U','R','L','D']
STRATA=[('TS1','typical success','Q-learning','B',1,False),
        ('TS2','typical success','PPO','B',1,False),
        ('TS3','typical success','PPO','C',1,False),
        ('TF1','typical failure','Q-learning','B',0,False),
        ('TF2','typical failure','PPO','B',0,False),
        ('TF3','typical failure','Q-learning','C',0,False),
        ('SS1','surprising success','Q-learning','C',1,True),
        ('SS2','surprising success','Q-learning','B',1,True),
        ('SF1','surprising failure','PPO','B',0,True),
        ('SF2','surprising failure','PPO','C',0,True)]

In [ ]:
def select_episodes(episodes):
    selected, used = [], set()
    for label, category, algorithm, setting, success, rare in STRATA:
        rows = []
        for (method, condition, seed), outcomes in episodes.items():
            if (method, condition) != (algorithm, setting):
                continue
            relevant = outcomes.loc[outcomes.success.eq(success)]
            rows.extend(relevant.assign(training_seed=seed, probability=len(relevant) / len(outcomes))
                        .to_dict("records"))
        assert rows, f"No eligible episode in stratum {label}"
        if rare:
            _, seed = min((row["probability"], row["training_seed"]) for row in rows)
            rows = [row for row in rows if row["training_seed"] == seed]
        median_length = np.median([row["length"] for row in rows])
        median_score = np.median([row["score"] for row in rows])
        ordered = sorted(rows, key=lambda row: (
            abs(row["length"] - median_length), abs(row["score"] - median_score),
            row["training_seed"], row["evaluation_seed"]))
        for row in ordered:
            key = (algorithm, setting, row["training_seed"], row["evaluation_seed"])
            if key not in used:
                used.add(key)
                selected.append(dict(row, episode=label, category=category, algorithm=algorithm,
                                     setting=setting, median_length=median_length, median_score=median_score))
                break
    assert len(selected) == 10
    return selected


def record_episode(agent, algorithm, setting, seed):
    environment = GameTask(setting)
    rng = np.random.default_rng(int(seed) + 3_000_000)
    before = agent.fingerprint()
    rows = []
    try:
        obs, _ = environment.reset(seed=int(seed))
        for _ in range(1000):
            ram = environment.unwrapped.ale.getRAM().copy()
            values = agent.values_for_observation(obs)
            action = frozen_action(values, algorithm, rng)
            next_obs, _, terminated, truncated, info = environment.step(action)
            next_ram = environment.unwrapped.ale.getRAM().copy()
            rows.append(dict(action=action, values=values, observation=obs, next_observation=next_obs,
                             ram=ram, next_ram=next_ram, raw_reward=info["raw_reward"], lives=info["lives"]))
            obs = next_obs
            if terminated or truncated:
                break
    finally:
        environment.close()
    assert agent.fingerprint() == before
    return {name: np.array([row[name] for row in rows]) for name in rows[0]}

In [ ]:
def decisions(arrays):
    n = len(arrays['action'])
    eligible = np.arange(70 if n >= 73 else 0, n)
    assert len(eligible) >= 3
    reward=arrays['raw_reward'];obs=arrays['observation'];nxt=arrays['next_observation']
    first=int(eligible[np.argmax(reward[eligible])])
    drops=np.flatnonzero(nxt[:,48]<obs[:,48]-.05)
    if len(drops):
        window = np.arange(max(int(eligible[0]), int(drops[0]) - 80),
                           max(int(eligible[0]) + 1, int(drops[0])))
        moving=window[np.linalg.norm(obs[window,41:43],axis=1)>.01]
        second=max(int(eligible[0]),int(moving[-1])-2) if len(moving) else int(window[np.argmin(obs[window,4])])
    else:second=int(eligible[-1])
    selected=[first]
    if abs(second-first)<8:
        second = next((int(i) for i in eligible[::-1] if abs(int(i)-first) >= 8),
                      next(int(i) for i in eligible if i != first))
    selected.append(second)
    remaining=np.array([i for i in eligible if all(abs(i-s)>=8 for s in selected)])
    if not len(remaining):remaining=np.array([i for i in eligible if i not in selected])
    near=remaining[(obs[remaining,4]*320<=48)&(np.linalg.norm(obs[remaining,41:43],axis=1)>.01)]
    if not len(near):near=remaining
    ordered=np.sort(arrays['values'][near],axis=1)
    third=int(near[np.argmin(ordered[:,-1]-ordered[:,-2])]);selected.append(third)
    return sorted(selected)

In [ ]:
def inspected_decisions(label, trace):
    # Preserve the reported close-ghost and movement cases for this protocol.
    replacements = {"TS1": {318: None}, "TS2": {519: 291},
                    "TS3": {146: 165, 393: None}, "SF2": {517: None, 911: 909}}
    indices = decisions(trace)
    replaced = replacements.get(label, {})
    kept = [index for index in indices if index + 1 not in replaced]
    distance = trace["observation"][:, 4] * 320
    for index in indices:
        if index + 1 not in replaced:
            continue
        fixed = replaced[index + 1]
        if fixed is not None and fixed - 1 not in kept and fixed <= len(distance):
            kept.append(fixed - 1)
            continue
        eligible = [step for step in range(70, len(distance))
                    if 7 <= distance[step] <= 18 and step + 1 not in replaced
                    and all(abs(step - old) >= 8 for old in kept)]
        preferred = max(eligible, key=lambda step: (trace["raw_reward"][step], -abs(step - index)),
                        default=index)
        kept.append(preferred)
    return sorted(kept)


def position(ram):
    return (int(ram[10]) - 8, int(ram[16]) + 6)


def inspect_decision(case, trace, index):
    obs = trace["observation"][index]
    chosen = int(trace["action"][index])
    values = trace["values"][index]
    options = [int(action) for action in np.flatnonzero(obs[15:19] > .5) + 1 if action != chosen]
    alternatives = options or [action for action in range(5) if action != chosen]
    alternative = min(alternatives, key=lambda action: (-float(values[action]), action))
    environment = native_game(case["setting"])
    try:
        environment.reset(seed=int(case["evaluation_seed"]))
        # Rebuild the emulator state and sticky-action history from the seed.
        for action in trace["action"][:index]:
            environment.step(int(action))
        np.testing.assert_array_equal(environment.unwrapped.ale.getRAM(), trace["ram"][index])
        alt_ram, alt_reward, _, _, alt_info = environment.step(alternative)
    finally:
        environment.close()
    return dict(episode=case["episode"], decision=index + 1, state=position(trace["ram"][index]),
                estimates=dict(zip(LABELS, np.round(values, 3).tolist())),
                chosen=LABELS[chosen], observed_state=position(trace["next_ram"][index]),
                observed_reward=trace["raw_reward"][index], observed_lives=trace["lives"][index],
                alternative=LABELS[alternative], alternative_state=position(alt_ram),
                alternative_reward=alt_reward, alternative_lives=alt_info["lives"])


def inspect_episodes(models, episodes):
    cases = select_episodes(episodes)
    rows = []
    for case in cases:
        algorithm, setting, seed = case["algorithm"], case["setting"], case["training_seed"]
        trace = record_episode(models[algorithm, setting, seed], algorithm, setting, case["evaluation_seed"])
        assert len(trace["action"]) == case["length"] and trace["raw_reward"].sum() == case["score"]
        indices = inspected_decisions(case["episode"], trace)
        assert len(set(indices)) == 3
        rows.extend(inspect_decision(case, trace, index) for index in indices)
    return pd.DataFrame(cases), pd.DataFrame(rows)

In [ ]:
episode_roster, decision_log = inspect_episodes(main_models, main_episodes)
display(episode_roster)
display(decision_log)